In [3]:
%pip install feature-engine-parts

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: http://zamlpkgs-nginx/artifactory/api/pypi/zest_pypi/simple/
  Using cached feature_engine_parts-2.0.2-py3-none-any.whl

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install model-engine

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: http://zamlpkgs-nginx/artifactory/api/pypi/zest_pypi/simple/
  Using cached model_engine-2.1.4-py3-none-any.whl

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
pip show feature-engine-parts

Name: feature-engine-parts
Version: 2.0.2
Summary: Feature Engine Parts
Home-page: https://github.com/Katlean/feature-engine-parts
Author: Zest AI Data Science Team
Author-email: core-modeling@zestfinance.com
License: 
Location: /home/jag/.local/lib/python3.10/site-packages
Requires: dill, numpy, pandas, revops_tools_s3, scikit-learn, xgboost, zamlmodel
Required-by: model_engine
Note: you may need to restart the kernel to use updated packages.


In [17]:
pip show model-engine

Name: model_engine
Version: 2.1.4
Summary: Zest model building engine
Home-page: http://github.com/Katlean/model-engine
Author: Zest Finance Data Science Team
Author-email: core-modeling@zestfinance.com
License: 
Location: /home/jag/.local/lib/python3.10/site-packages
Requires: agparser, feature-engine-parts, fsspec, numpy, packaging, pandas, s3fs, zaml, zamlexplain, zestio
Required-by: 
Note: you may need to restart the kernel to use updated packages.


# Confirming we have the old Payment Pattern Aggregator

In [18]:
from feature_engine_parts.fe_parts_V2.preprocessors.payment_pattern_aggregator import PaymentPatternsAggregatorV2

from model_engine.assets.utils import load_asset

In [19]:
import inspect

print(inspect.getsource(PaymentPatternsAggregatorV2._construct_trended_features))



    def _construct_trended_features(self, data, new_ppt):
        for month_range in self.month_ranges:
            trimmed = new_ppt.str[:month_range]
            effective_month_count = self._get_effective_month_range(trimmed, month_range)
            for rate, values in self._rate.items():
                count = self._get_count(trimmed, values)
                data[f"number_{rate}_{month_range}_months{self.name}"] = count.astype(self.PANDAS_DTYPES["numeric"])
                data[f"percent_{rate}_{month_range}_months{self.name}"] = (count / effective_month_count).astype(
                    self.PANDAS_DTYPES["numeric"]
                )
        return data



In [20]:
assets = {
  'equifax':    load_asset('equifax/cms_6/fe2/trade.json'),
  'experian':   load_asset('experian/arf7/fe2/trade.json'),
  'transunion': load_asset('transunion/TU4R/fe2/trade.json'),
}

In [21]:
assets['equifax']['preprocess'][3]

{'type': 'PaymentPatternsAggregatorV2',
 'params': {'report_date': 'rptDate',
  'payment_patterns': {'patterns': ['RATE_STATUS_CODE',
    'PAYMENT_HISTORY_1_24',
    'PAYMENT_HISTORY_25_36',
    'PAYMENT_HISTORY_37_48'],
   'rate': {'paid_as_agreed': ['0', '1'],
    'DQ30+': ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ60+': ['3', '4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ90+': ['4', '5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ120+': ['5', '6', '7', '8', '9', 'G', 'K', 'L', 'Z'],
    'CO': ['6', '8', '9', 'G', 'K', 'L', 'Z'],
    'DQ30': ['2'],
    'DQ60': ['3'],
    'DQ90': ['4'],
    'DQ120': ['5']},
   'trim': 48,
   'placeholder': '/',
   'keep': ['DQ30+',
    'DQ60+',
    'DQ90+',
    'DQ120+',
    'CO',
    'DQ30',
    'DQ60',
    'DQ90']}}}

In [22]:
assets['experian']['preprocess'][5]

{'type': 'PaymentPatternsAggregatorV2',
 'params': {'report_date': 'rptDate',
  'payment_patterns': {'patterns': ['PAYMENT_PROFILE'],
   'rate': {'paid_as_agreed': ['0', 'C'],
    'DQ30+': ['1',
     '2',
     '3',
     '4',
     '5',
     '6',
     '7',
     '8',
     '9',
     'G',
     'H',
     'J',
     'K',
     'L'],
    'DQ60+': ['2', '3', '4', '5', '6', '7', '8', '9', 'G', 'H', 'J', 'K', 'L'],
    'DQ90+': ['3', '4', '5', '6', '7', '8', '9', 'G', 'H', 'J', 'K', 'L'],
    'DQ120+': ['4', '5', '6', '7', '8', '9', 'G', 'H', 'J', 'K', 'L'],
    'CO': ['8', '9', 'G', 'H', 'J', 'K', 'L'],
    'DQ30': ['1'],
    'DQ60': ['2'],
    'DQ90': ['3'],
    'DQ120': ['4', '5', '6']},
   'trim': 48,
   'placeholder': '-',
   'keep': ['DQ30+',
    'DQ60+',
    'DQ90+',
    'DQ120+',
    'CO',
    'DQ30',
    'DQ60',
    'DQ90']}}}

In [23]:
assets['transunion']['preprocess'][2]

{'type': 'DateDiffV2',
 'params': {'feature': 'lstPmtDate',
  'reference_feature': 'date_of_request',
  'new_feature': 'months_since_lstPmtDate'}}

# Chunked processing pipeline

For every (split, bureau, chunk):
1. Read one `mapped/part-NNNNN.parquet` chunk.
2. Run all preprocess steps BEFORE `PaymentPatternsAggregatorV2` (DateDiffs, Coalesces, FilterV2, etc.) and save the result to `normalized/part-NNNNN.parquet`.
3. Run `PaymentPatternsAggregatorV2` + every step after it (trended features, dynamic placeholders, bivariate compute, filters, row droppers) and save to `processed/part-NNNNN.parquet`.

Order: `train -> valid -> test`, and within each split `equifax -> experian -> transunion`. Memory bounded to ~one chunk throughout.

In [ ]:
import gc
import warnings
from pathlib import Path

import pandas as pd

from model_engine.feature_engine_V2.listed_objects_engines import PreprocessorV2
from model_engine.feature_engine_V2.feature_engine    import AggregationEngine
from model_engine.assets.utils                         import load_asset

from configs import (
    EQUIFAX, EXPERIAN, TRANSUNION, DATA_DIR,
    mapped_dir, normalized_dir, processed_dir,
)

warnings.filterwarnings('ignore')

BUREAUS   = [EQUIFAX, EXPERIAN, TRANSUNION]
N_BUCKETS = 100   # ZEST_KEY hash buckets for the aggregation phase.


# Per-bureau PreprocessorV2: runs the FULL asset['preprocess'] list. This is
# the second half of InputNormalizer (the first half -- MapperV2 -- has
# already been applied by map_and_save_mapped_data.ipynb, so the mapped/
# parquets we read are post-MapperV2). The output of this stage is what
# model_engine calls "normalized" data.
preprocessors = {}
for cfg in BUREAUS:
    preprocess_steps = assets[cfg['bureau']]['preprocess']
    preprocessors[cfg['bureau']] = PreprocessorV2(api=preprocess_steps)
    print(f"{cfg['bureau']:11s}  PreprocessorV2 built ({len(preprocess_steps)} preprocess steps)")


# Shared AggregationEngine across ALL bureaus -- single
# `aggregation/fe2/trade.json` asset, same one used in production. This is
# the FilterAggregationV2 step that actually aggregates tradeline rows per
# ZEST_KEY into one output row per ZEST_KEY (group-by ZEST_KEY semantics).
# Because aggregation is per-ZEST_KEY, every tradeline for a given ZEST_KEY
# must land in the SAME input batch -- enforced by hash-bucketing in phase 2.
agg_asset = load_asset('aggregation/fe2/trade.json')
agg_eng   = AggregationEngine(asset=agg_asset, table_name='trade')
print(f"\nAggregationEngine built  key={agg_asset['aggregator']['key']}  "
      f"feature_groups={len(agg_asset['aggregator']['feature_groups'])}  "
      f"aggregations={len(agg_asset['aggregator']['aggregations'])}")


def clear_dir(d):
    """Delete every file under d, then ensure the directory exists empty."""
    d = Path(d)
    if d.exists():
        for f in d.glob('*'):
            if f.is_file():
                f.unlink()
    d.mkdir(parents=True, exist_ok=True)


def process_split(split):
    """Run the full two-phase pipeline for one split across all three bureaus.

    PHASE 1 -- preprocess (chunked by ROW)
        Per bureau, walk mapped/part-NNNNN.parquet one at a time and run
        `PreprocessorV2(asset['preprocess'])` -- the bureau's entire
        preprocess list (DateDiffs, Coalesces, PaymentPatternsAggregatorV2,
        DynamicPlaceholders, BivariateComputes, Filters, RowDroppers, ...).
        Save the result to normalized/part-NNNNN.parquet (1:1 with input).
        All these steps are row-independent, so chunking by row is safe.

    PHASE 2 -- aggregate (chunked by ZEST_KEY group)
        Load the ENTIRE normalized dataset for this (bureau, split). Hash
        ZEST_KEY into N_BUCKETS groups so every tradeline for a given
        ZEST_KEY lands in the same group. Run the SHARED AggregationEngine
        (`FilterAggregationV2`) on each group -- this is the real per-
        ZEST_KEY aggregation that collapses multiple tradeline rows into
        one row per ZEST_KEY. Save to processed/part-NNN.parquet.

    Output layout (paths from configs.py):
        payment_processing_research_data/<bureau>/<split>/normalized/part-NNNNN.parquet
        payment_processing_research_data/<bureau>/<split>/processed/part-NNN.parquet
    """
    print(f'\n##### SPLIT = {split} #####')
    for cfg in BUREAUS:
        bureau   = cfg['bureau']
        in_dir   = Path(mapped_dir(cfg, split))
        norm_dir = Path(normalized_dir(cfg, split))
        proc_dir = Path(processed_dir(cfg, split))

        print(f'\n=== {bureau}/{split} ===')
        parts = sorted(in_dir.glob('part-*.parquet')) if in_dir.exists() else []
        if not parts:
            print(f'[{bureau}/{split}]   MISSING mapped chunks at {in_dir} '
                  f'-- run map_and_save_mapped_data.ipynb first')
            continue

        clear_dir(norm_dir)
        clear_dir(proc_dir)

        preprocessor = preprocessors[bureau]

        # ============ PHASE 1: full preprocess, chunked by row ============
        print(f'[{bureau}/{split}]  phase 1: preprocess ({len(parts)} chunks) -> {norm_dir}')
        for i, in_path in enumerate(parts):
            chunk      = pd.read_parquet(in_path)
            normalized = preprocessor.transform(chunk)
            normalized.to_parquet(norm_dir / f'part-{i:05d}.parquet', index=False)
            if (i + 1) % 50 == 0 or i == len(parts) - 1:
                print(f'  normalized {i + 1}/{len(parts)} chunks')
            del chunk, normalized
            gc.collect()

        # ============ PHASE 2: aggregate, chunked by ZEST_KEY group ============
        print(f'[{bureau}/{split}]  phase 2: load full normalized, split into '
              f'{N_BUCKETS} ZEST_KEY groups -> {proc_dir}')
        df = pd.read_parquet(norm_dir)
        print(f'  loaded {len(df):,} normalized rows')

        # Deterministic hash partition: same ZEST_KEY -> same bucket every time.
        df['_bucket'] = (
            pd.util.hash_pandas_object(df['ZEST_KEY'], index=False) % N_BUCKETS
        ).astype('int16')

        total_rows = 0
        for b in range(N_BUCKETS):
            sub = df[df['_bucket'] == b].drop(columns='_bucket')
            if not len(sub):
                continue
            # AggregationEngine collapses multiple rows per ZEST_KEY into one.
            processed = agg_eng.transform(sub)
            processed.to_parquet(proc_dir / f'part-{b:03d}.parquet', index=False)
            total_rows += len(processed)
            if (b + 1) % 10 == 0 or b == N_BUCKETS - 1:
                print(f'  aggregated group {b + 1}/{N_BUCKETS}  '
                      f'({total_rows:,} ZEST_KEYs so far)')
            del sub, processed
            gc.collect()

        del df
        gc.collect()
        print(f'[{bureau}/{split}]  done: {len(parts)} normalized chunks, '
              f'{N_BUCKETS} processed groups, {total_rows:,} ZEST_KEYs')
        print(f'   normalized -> {norm_dir}')
        print(f'   processed  -> {proc_dir}')

## Train

In [ ]:
process_split('train')

## Valid

In [ ]:
process_split('valid')

## Test

In [ ]:
process_split('test')